# w2 · LSTM vs GRU for behaviour trajectories (PyTorch)

`w2_neural_models.ipynb` classifies churn from the 90-day sequence. This notebook uses the **same tensor, split and scalers** for a different task: predicting how each user will behave over the **next 7 days**. The goal is a behaviour simulator that the retention agents of `wfinal` can query.

The 90-day window is split in two:

```text
day:   0 ............................. 82 | 83 ........ 89
       input (83 days, 20 channels)       | target (7 days, 13 event channels)
```

| Cohort | Input | Target |
|---|---|---|
| `train_feb` | 2016-11-03 to 2017-01-24 | 2017-01-25 to 2017-01-31 |
| `test` (March) | 2016-12-01 to 2017-02-21 | 2017-02-22 to 2017-02-28 |

The days are "future" relative to what the model sees, not to the calendar, so the export does not need to be rebuilt.

**Option B.** The network predicts only the **events** (listening and payments). The 7 subscription-state channels are not predicted: they are rebuilt from the predicted events with the export's rules, so a simulated week can never contradict itself (for example, a payment that does not reset `days_since_last`).

The two recurrent cells are compared under identical conditions: same input, head, loss, search space, budget and evaluation.

## 1. Configuration

Settings live in `src/models/trajectory_models/config.py`. Artifacts are written to `artifacts/sherlock_w2_traj/`, so the classifier artifacts (`artifacts/sherlock_w2/`, used by the report and by `wfinal`) are not touched.

- `N_TRIALS=10` per model, `EPOCHS=40`, `PATIENCE=5`, `BATCH_SIZE=512`: same procedure as the classifier, with twice the search budget because the task is new.
- `HIDDEN_DIMS=(64, 128, 256)`: wider than the classifier, since the head now emits 7 × 13 = 91 values instead of 1.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

warnings.filterwarnings("ignore")

from src.models.trajectory_models.config import (
    BATCH_SIZE,
    COMPARISON_DIR,
    EPOCHS,
    EXPORT_DIR,
    FEATURE_DIR,
    HIDDEN_DIMS,
    HORIZON,
    INPUT_DAYS,
    LAYER_DIM_RANGE,
    LR_RANGE,
    N_TARGETS,
    N_TRIALS,
    PATIENCE,
    SEED,
    TARGET_NAMES,
    WEIGHT_DECAY_RANGE,
    model_dirs,
)
from src.models.trajectory_models.model import MODELS
from src.models.trajectory_models.train import resolve_device

DEVICE = resolve_device()

settings = pd.DataFrame(
    [
        ("models", ", ".join(name.upper() for name in MODELS)),
        ("export_dir", str(EXPORT_DIR)),
        ("feature_dir", str(FEATURE_DIR)),
        ("comparison_dir", str(COMPARISON_DIR)),
        ("device", str(DEVICE)),
        ("seed", SEED),
        ("input_days", INPUT_DAYS),
        ("horizon", HORIZON),
        ("n_targets", N_TARGETS),
        ("n_trials", N_TRIALS),
        ("epochs", EPOCHS),
        ("patience", PATIENCE),
        ("batch_size", BATCH_SIZE),
        ("hidden_dims", HIDDEN_DIMS),
        ("layer_dim_range", LAYER_DIM_RANGE),
        ("lr_range", LR_RANGE),
        ("weight_decay_range", WEIGHT_DECAY_RANGE),
    ],
    columns=["setting", "value"],
)
display(settings)
pd.DataFrame({name.upper(): {k: str(v) for k, v in model_dirs(name).items()} for name in MODELS})

## 2. Data

### 2.1 Same export, same split, same scalers

`load_trajectory_data` loads the artifacts of the classifier (`load_prepared_data`) and only changes what each batch contains. The split (`fit` / `stop` / `tune` from February, `test` = March, `development` = all of February) and the scalers fitted per phase are reused unchanged.

In [ ]:
from src.models.trajectory_models.data import load_trajectory_data

data = load_trajectory_data(feature_dir=FEATURE_DIR, export_dir=EXPORT_DIR)

print("tensor:", data.view.shape, data.view.dtype)
print("input_dim:", data.input_dim)
pd.DataFrame(
    [{"part": part, "n_users": len(data.rows(part))}
     for part in ("fit", "stop", "tune", "test", "development")]
)

### 2.2 Targets

Each batch is `x` of shape `(B, 83, 20)` (scaled days 0..82) and `y` of shape `(B, 7, 13)` (days 83..89):

| Positions | Target | Type | Defined on |
|---|---|---|---|
| 0–6 | `num_25` … `total_secs` | scaled amount | active days |
| 7 | `active` | 0/1 | every day |
| 8 | `pay` (`tx_count > 0`) | 0/1 | every day |
| 9–10 | `tx_paid_sum`, `tx_plan_days_sum` | scaled amount | payment days |
| 11–12 | `cancel`, `auto_renew` | 0/1 | payment days |

Binary targets are read from the **raw** tensor: after standard scaling, a raw `tx_count` of 1 becomes about −0.1, not 1.

Listening is split in two parts ("hurdle"): *does the user listen?* and *how much, if they do?*. After scaling, an inactive day and an average active day are both 0, so a single regression over all days would learn the average of the two and produce users who listen a little every day, which never happens.

In [ ]:
xb, yb = next(iter(data.iter_batches(data.rows("fit")[:BATCH_SIZE], "fit", BATCH_SIZE)))
print("x:", tuple(xb.shape), "| y:", tuple(yb.shape))

y_np = yb.numpy()
pd.DataFrame(
    {
        "target": TARGET_NAMES,
        "mean (batch)": y_np.reshape(-1, N_TARGETS).mean(axis=0).round(4),
    }
).set_index("target").T

## 3. Model and loss

### 3.1 Architecture

Same encoder as the classifier (stacked LSTM or GRU, last hidden state, `BatchNorm1d`). Only the head changes: `Linear(hidden, 7 · 13)` reshaped to `(B, 7, 13)`. All 7 days are emitted at once (direct multi-horizon). With H = 7 an autoregressive decoder would add error accumulation without a clear benefit.

```text
(B, 83, 20) → LSTM/GRU × layer_dim → h_83 → BatchNorm → Linear(H, 91) → (B, 7, 13)
                                                             ├─ 7 amounts (scaled)
                                                             └─ 6 binary logits
```

In [ ]:
for name, model_cls in MODELS.items():
    example = model_cls(input_dim=data.input_dim, hidden_dim=64, layer_dim=2).eval()
    with torch.no_grad():
        out = example(xb)
    n_params = sum(p.numel() for p in example.parameters())
    print(f"{name.upper():4s} | output {tuple(out.shape)} | parameters (H=64, 2 layers): {n_params:,}")

### 3.2 Composite loss

```text
L = w_active · BCE(active)
  + w_listen · Huber(listening | active day)
  + w_pay    · BCE(pay)
  + w_amount · Huber(paid amount, plan days | payment day)
  + w_flags  · BCE(cancel, auto-renew | payment day)
```

- **No `pos_weight` on `pay`.** The simulator samples payments from `sigmoid(logit)`, so the probabilities must stay calibrated. Up-weighting the positives would make every simulated user pay too often.
- **Fixed weights.** Each term is divided by the loss of a constant predictor (prevalence or mean) computed on a sample of `fit`. Every term then starts near 1, and Optuna cannot lower the objective by down-weighting the hard term. The weights are computed once and shared by both models.

In [ ]:
from src.models.trajectory_models.loss import calibrate_weights

loss_weights, baseline_terms = calibrate_weights(data, data.rows("fit"), phase="fit")

ARTIFACTS_DIR = COMPARISON_DIR.parent
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "loss_weights.json").write_text(
    json.dumps({"weights": loss_weights, "constant_predictor_loss": baseline_terms}, indent=2)
)
pd.DataFrame({"constant-predictor loss": baseline_terms, "weight": loss_weights}).round(4)

### 3.3 Optimisation and search

Same procedure as the classifier: AdamW, gradient clipping at 1.0, AMP on GPU, at most 40 epochs, early stopping on `stop` with `patience=5` and best-epoch restore.

For each family, Optuna (TPE, same seed, 10 trials) minimises the **weighted composite loss on `tune`**. There is no threshold search.

In [ ]:
from src.models.trajectory_models.tune import tune_model

best = {}
for name in MODELS:
    print(f"\n===== {name.upper()} =====")
    best[name] = tune_model(
        data,
        loss_weights=loss_weights,
        model_name=name,
        n_trials=N_TRIALS,
        epochs=EPOCHS,
        patience=PATIENCE,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        seed=SEED,
    )

pd.DataFrame(
    [
        {
            "model": name.upper(),
            **best[name]["best_params"],
            "parameters": best[name]["n_parameters"],
            "best_epoch": best[name]["best_epoch"],
            "tune_loss": best[name]["best_value"],
        }
        for name in MODELS
    ]
).set_index("model")

### 3.4 Training history

`train_loss` vs `stop_loss` of the best trial of each family. The dotted line marks the epoch restored by early stopping.

In [ ]:
from src.models.trajectory_models.evaluation import MODEL_COLORS

COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, len(MODELS), figsize=(11, 3.8), sharey=True)
for ax, name in zip(np.atleast_1d(axes), MODELS):
    history = pd.DataFrame(best[name]["history"])
    ax.plot(history["epoch"], history["train_loss"], color=MODEL_COLORS[name], label="train")
    ax.plot(history["epoch"], history["stop_loss"], color=MODEL_COLORS[name], ls="--", label="stop")
    ax.axvline(best[name]["best_epoch"], color="grey", ls=":")
    ax.set_title(name.upper())
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("weighted composite loss")
np.atleast_1d(axes)[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(COMPARISON_DIR / "training_history.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Test

Each tuned architecture is refitted on all of February (`development`) for its `best_epoch`, then predicts March (days 83..89) once. A checkpoint is saved to `artifacts/sherlock_w2_traj/<model>/checkpoints/<model>_traj.pt`, with the parameters, the loss weights, the window sizes and the scaler phase (`development`). That is all an environment needs to reload it.

In [ ]:
from src.models.trajectory_models.test import test_model

results = {}
for name in MODELS:
    print(f"\n===== {name.upper()} =====")
    results[name] = test_model(best[name], data, loss_weights=loss_weights, device=DEVICE)

preds = {name: results[name]["pred"] for name in MODELS}
TEST_ROWS = data.rows("test")
SCALER = data.states["development"]

pd.DataFrame(
    [
        {
            "model": name.upper(),
            "epochs": results[name]["epochs"],
            "parameters": results[name]["n_parameters"],
            "seconds_per_epoch": round(results[name]["seconds_per_epoch"], 2),
            "checkpoint": str(results[name].get("checkpoint", "")),
        }
        for name in MODELS
    ]
).set_index("model")

## 5. Evaluation

The error alone does not say whether a model is useful, so every group is compared with a predictor that learns nothing. All numbers are in the raw encoding: listening in `log1p`, amounts in NTD and days.

| Group | Metric | Baselines |
|---|---|---|
| Listening (0–6) | MAE, RMSE, skill = 1 − MAE / MAE(persistence) | persistence (day 82), mean of days 76–82, previous week |
| `active` | AUC, AP, Brier | active on day 82, active share of days 76–82 |
| `pay` (daily and weekly) | AUC, AP, Brier | expiry rule: an auto-renew user pays when the membership expires |
| Amount and plan | MAE on real payment days | last amount paid |

The model's point forecast for listening is `p(active) × amount if active`.

In [ ]:
from src.models.trajectory_models.evaluation import evaluate_trajectories

report = evaluate_trajectories(preds, data, TEST_ROWS, scaler_state=SCALER)
for key in ("listening", "horizon", "channel", "events", "amounts", "realism", "state"):
    report[key].to_csv(COMPARISON_DIR / f"{key}.csv")

print("Listening (log1p)")
display(report["listening"].round(4))
print("Events")
display(report["events"].round(4))
print("Paid amount and plan days on real payment days")
display(report["amounts"].round(3))

### 5.1 Error by horizon and by channel

The error should grow with h. A flat curve means the model is predicting something close to a constant.

In [ ]:
from src.models.trajectory_models.evaluation import plot_horizon

plot_horizon(report, save_path=COMPARISON_DIR / "horizon_mae.png")
plt.show()
display(report["horizon"].round(4))
display(report["channel"].round(4))

### 5.2 Example trajectories

Real `total_secs` (last 4 weeks of input + the 7 target days) against each model's expected forecast, for active users, users who fade out and inactive users.

In [ ]:
from src.models.trajectory_models.evaluation import pick_example_users, plot_example_trajectories

examples = pick_example_users(data, TEST_ROWS)
plot_example_trajectories(
    preds, data, TEST_ROWS, examples, scaler_state=SCALER,
    save_path=COMPARISON_DIR / "example_trajectories.png",
)
plt.show()

## 6. Option B: simulated weeks

For the agent, the model is used as a **simulator**: `active`, `pay`, cancel and auto-renew are sampled from their probabilities, the amounts are written on the sampled days, and the 7 state channels are rebuilt with the export's rules (`simulate.py`).

**Realism.** A simulator with a good MAE can still produce unrealistic users. The share of active days, the share of users who pay during the week and the distribution of `total_secs` should match the real March week.

**Rebuilt state.** Three sources are compared against the real state channels:

- `no_payment_rule`: nobody pays, the counters only advance by one day (lower reference).
- `oracle_events`: the **real** events go through the same rebuild. Its error is the cost of the expiry rule alone (`max(days_to_expire − 1, 0) + plan_days`; the export uses the real expiry date).
- `LSTM` / `GRU`: events sampled from each model.

In [ ]:
from src.models.trajectory_models.evaluation import plot_secs_distribution

print("Realism of one sampled week per user")
display(report["realism"].round(4))
print("Error of the rebuilt state (days, on days where the real state is known)")
display(report["state"].round(4))

plot_secs_distribution(report, save_path=COMPARISON_DIR / "secs_distribution.png")
plt.show()

### 6.1 Using the simulator

`simulate_next_week` takes the **raw** last 83 days of a set of users, `(N, 83, 20)` in the export encoding, and returns one sampled week, `(N, 7, 20)`, in the same encoding. `rollout` chains several weeks by appending each simulated week and dropping the oldest 7 days. An environment that applies an action to a user (adds logs or a transaction) only needs to rebuild the window and call these functions.

Limitation: the model was trained on data without retention offers. It does not learn how a user responds to an offer. It only projects forward the change an action makes to the user's history.

In [ ]:
from src.models.trajectory_models.simulate import rollout

sample_rows = TEST_ROWS[:5]
window = data.raw(sample_rows)[:, -INPUT_DAYS:, :]
ix = {spec["name"]: i for i, spec in enumerate(data.channel_specs)}

for name in MODELS:
    weeks = rollout(results[name]["model"].cpu(), window, SCALER, data.channel_specs, n_weeks=4, seed=SEED)
    print(f"{name.upper()}: simulated tensor {weeks.shape}")
    print(pd.DataFrame(
        {
            "active_days": (weeks[:, :, ix["active"]] > 0).sum(axis=1),
            "payments": (weeks[:, :, ix["tx_count"]] > 0).sum(axis=1),
            "days_to_expire_end": np.rint(
                np.sign(weeks[:, -1, ix["tx_days_to_expire"]])
                * np.expm1(np.abs(weeks[:, -1, ix["tx_days_to_expire"]]))
            ).astype(int),
        },
        index=[f"user {i}" for i in range(len(sample_rows))],
    ).T)

## 7. LSTM vs GRU

The per-user listening MAE of both models is compared with a paired bootstrap: the same resampled users score both models, so the interval of the difference accounts for their correlation. If the interval contains 0, the smaller and faster model is preferable.

In [ ]:
from src.models.trajectory_models.evaluation import paired_bootstrap_mean

names = list(MODELS)
boot = paired_bootstrap_mean(
    report["per_user_listen_mae"][names[0]],
    report["per_user_listen_mae"][names[1]],
    n_boot=1000,
    seed=SEED,
)
print(f"Per-user listening MAE, difference = {names[0].upper()} - {names[1].upper()}")
display(pd.Series(boot).to_frame("value"))

summary = pd.DataFrame(
    {
        name.upper(): {
            "listening MAE": report["listening"].loc[name.upper(), "MAE"],
            "skill vs persistence": report["listening"].loc[name.upper(), "skill_vs_persistence"],
            "active AUC": report["events"].loc[("active", name.upper()), "AUC"],
            "pay AP": report["events"].loc[("pay", name.upper()), "AP"],
            "weekly pay AP": report["events"].loc[("pay_week", name.upper()), "AP"],
            "parameters": results[name]["n_parameters"],
            "seconds/epoch": results[name]["seconds_per_epoch"],
        }
        for name in names
    }
)
summary.to_csv(COMPARISON_DIR / "summary.csv")
summary

## 8. Artifacts and limitations

```text
artifacts/sherlock_w2_traj/
├── loss_weights.json             # fixed loss weights (shared by both models)
├── lstm/ , gru/
│   ├── optuna/                   # study, trials, best params, best-trial history
│   └── checkpoints/<model>_traj.pt
└── comparison/                   # tables (csv) and figures
```

Limitations:

- **Not comparable with the classifier.** The classifier predicts churn (log-loss, AUC); this notebook predicts trajectories (MAE, AUC of daily events). They are different tasks.
- **No response to offers.** The training data contain no retention actions; the simulator only projects the history an action leaves behind.
- **Approximate expiry.** The rebuilt `days_to_expire` uses the predicted plan days instead of the real expiry date; `oracle_events` measures that error.
- **Amounts without noise.** The sampled weeks use the predicted amount on each active or payment day, so their spread is narrower than the real one (see the `total_secs` distribution).
- **Shifted use.** Training uses days 83..89 of the window. In an environment the model is applied to the last 83 days before the cutoff. It has no calendar input, so this works, but it assumes behaviour is stable over time.